In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
import torch
import evaluate
import numpy as np
from peft import UIOrthoLoRAConfig, UILinLoRAConfig, get_peft_model, TaskType, PeftConfig, PeftModel
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import load_dataset

/opt/anaconda3/envs/intlxAdv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/intlxAdv/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/opt/anaconda3/envs/intlxAdv/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-06-26 12:20:25.567045: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the followin

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def load_and_prepare(tokenizer, max_length=128):
    """Load E2E dataset and prepare tokenised fields."""
    ds = load_dataset("tuetschek/e2e_nlg")

    def linearise(record):
        mr = record["meaning_representation"]  # e.g. "name[Bibimbap House], food[Indian]"
        ref = record["human_reference"] if "human_reference" in record else record["reference"]
        prompt = f"{mr} => "  # simple prompt pattern
        example = prompt + ref
        tokenised = tokenizer(
            example,
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )
        labels = tokenised["input_ids"].copy()
        # Mask prompt tokens so they are ignored in the loss (label = -100)
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        prompt_len = len(prompt_ids)
        labels[:prompt_len] = [-100] * prompt_len
        tokenised["labels"] = labels
        return tokenised

    ds = ds.map(linearise, remove_columns=ds["train"].column_names)
    return ds

In [4]:
from pycocoevalcap.cider.cider import Cider

class CiderMetric:
    """Wraps pycocoevalcap so it looks like an `evaluate` metric."""
    def __init__(self):
        self.scorer = Cider()

    def compute(self, *, predictions, references):
        # pycocoevalcap expects dicts: {idx: ["sentence"]}
        hyps = {i: [pred] for i, pred in enumerate(predictions)}
        refs = {i: [ref]  for i, ref  in enumerate(references)}
        score, _ = self.scorer.compute_score(refs, hyps)
        return {"cider": score}

cider_metric = CiderMetric()

In [5]:
bleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
rouge_metric = evaluate.load("rouge")
nist_metric = evaluate.load("nist_mt")

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/guy.bilitski/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/guy.bilitski/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/guy.bilitski/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [6]:
def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels


def set_contiguous(model):
    for m in model.modules():
        if hasattr(m, "parametrizations") and "weight" in m.parametrizations:
            base = m.parametrizations.weight[0].base
            if not base.is_contiguous():
                base.data = base.data.contiguous()


# ------------------------------------------------------------------
# helper -----------------------------------------------------------
def _postprocess_strs(predictions, references):
    """Strip leading/trailing spaces & unify whitespace."""
    preds = [p.strip() for p in predictions]
    refs  = [r.strip() for r in references]
    return preds, refs
# ------------------------------------------------------------------



def compute_metrics(eval_pred):
    """Compute BLEU, METEOR, ROUGE-L, (optionally) NIST on E2E-NLG."""
    preds, labels = eval_pred

    # ── tensors → numpy ───────────────────────────────────────────────
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    # ── logits → ids if necessary ─────────────────────────────────────
    if preds.ndim == 3:                      # (batch, seq, vocab)
        preds = preds.argmax(-1)

    # ── un-mask labels ────────────────────────────────────────────────
    labels = labels.copy()
    labels[labels == -100] = tokenizer.pad_token_id

    # ── decode ────────────────────────────────────────────────────────
    pred_strs  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    label_strs = tokenizer.batch_decode(labels, skip_special_tokens=True)
    pred_strs  = [s.strip() for s in pred_strs]
    label_strs = [s.strip() for s in label_strs]

    # ── metrics ───────────────────────────────────────────────────────
    bleu   = bleu_metric.compute(
                predictions=pred_strs,
                references=[[r] for r in label_strs]
             )["score"]

    meteor = meteor_metric.compute(
                predictions=pred_strs,
                references=label_strs
             )["meteor"]

    rougeL = rouge_metric.compute(
                predictions=pred_strs,
                references=label_strs,
                use_stemmer=True
             )["rougeL"]

    cider = cider_metric.compute(
                predictions=pred_strs,
                references=label_strs   # wrapper handles dict-conversion
            )["cider"]

    # ---- NIST (may not exist on tiny samples) ------------------------
    nist_raw = nist_metric.compute(
                predictions=pred_strs,
                references=[[r] for r in label_strs]
              )
    nist_val = nist_raw.get("nist", nist_raw.get("score"))  # could be None

    # helper to round only numerics
    def _r(x):
        return round(float(x), 4) if isinstance(x, Number) else x

    out = {
        "bleu"  : _r(bleu),
        "meteor": _r(meteor),
        "rougeL": _r(rougeL),
        "cider"  : _r(cider),
    }
    if nist_val is not None:
        out["nist"] = _r(nist_val)

    return out


In [7]:
orthoLoRAConfig = UIOrthoLoRAConfig(
    target_modules=["attn.c_attn", "attn.c_proj"],
    fan_in_fan_out         = True,   # GPT-2 matrices are (out, in)
    initial_scaler         = 0.1,    # scale of the diagonal Σ at init
    initial_sigma          = 0.1,    # std-dev for the trainable Σ entries
    uiortholora_alpha      = 1,
    uiortholora_dropout    = 0,
    num_svalues_to_adapt   = 2,       # adapt the top-4 singular values
    num_svectors_to_adapt  = 2,       # adapt the corresponding vectors
    task_type              = TaskType.CAUSAL_LM
)

In [8]:
from peft import LoraConfig

lora_config = LoraConfig(
    target_modules=["attn.c_attn", "attn.c_proj"],
    r=2,                         # very low rank for easy debugging
    lora_alpha=1,               # no extra scaling
    lora_dropout=0.0,           # no dropout for deterministic behavior
    bias="none",                # keep bias untouched
    fan_in_fan_out=False,       # match GPT-2 shape: (out, in)
    task_type=TaskType.CAUSAL_LM
)


In [16]:
model_path = "gpt2-medium"
seed=42

torch.manual_seed(seed)
np.random.seed(seed)


tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = load_and_prepare(tokenizer)

base_model = AutoModelForCausalLM.from_pretrained(model_path)
base_model.config.pad_token_id = tokenizer.pad_token_id

Using the latest cached version of the module from /home/guy.bilitski/.cache/huggingface/modules/datasets_modules/datasets/tuetschek--e2e_nlg/bfeceb720929c2705bd227d1cfe5eaaab102a0bdac10dad618dac1e00c737430 (last modified on Sat Jun 21 16:13:44 2025) since it couldn't be found locally at tuetschek/e2e_nlg, or remotely on the Hugging Face Hub.


In [26]:
peft_config = PeftConfig.from_pretrained("outputs/models")

In [30]:
base_model_check = AutoModelForCausalLM.from_pretrained(peft_config.base_model_name_or_path)
base_model_check = base_model_check.to(device)

In [31]:
model = PeftModel.from_pretrained(base_model_check, "outputs/models")

In [ ]:
stop

In [18]:
base_model = base_model.to(device)


In [19]:
base_model.device

device(type='cuda', index=0)

In [12]:
stop

NameError: name 'stop' is not defined

In [20]:
model = get_peft_model(base_model, orthoLoRAConfig)

In [21]:
set_contiguous(model)
model.print_trainable_parameters()

trainable params: 147,936 || all params: 354,971,104 || trainable%: 0.0417


In [22]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="outputs/check",
    overwrite_output_dir=True,
    eval_strategy="no",
    save_strategy="no",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    eval_accumulation_steps=2,
    learning_rate=1e-3,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
)

In [23]:
trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"].select(range(500)),
        eval_dataset=ds["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [24]:
# trainer.train()
# from accelerate import Accelerator
# accelerator = Accelerator()
# trainer = accelerator.prepare(trainer)
trainer.train()


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,4.350600
100,4.257500
150,4.217100
200,4.127900
250,4.074500
300,4.017300


TrainOutput(global_step=315, training_loss=4.168453349764385, metrics={'train_runtime': 316.8974, 'train_samples_per_second': 7.889, 'train_steps_per_second': 0.994, 'total_flos': 580721971200000.0, 'train_loss': 4.168453349764385, 'epoch': 5.0})

In [25]:
trainer.save_model("outputs/models")

In [ ]:
stop

In [16]:
# Load the trained model from saved checkpoint
model = AutoModelForCausalLM.from_pretrained("outputs/models")
model = model.to("cuda")

In [18]:
# from pathlib import Path
# import json

# metrics = trainer.evaluate(ds["test"].select(range(100)))
# Path(training_args.output_dir).mkdir(parents=True, exist_ok=True)
# (Path(training_args.output_dir) / "test_metrics.json").write_text(json.dumps(metrics, indent=2))
# print("Test metrics saved to", training_args.output_dir)


KeyboardInterrupt: 

In [24]:
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


eval_loss: 4.2372
eval_model_preparation_time: 0.0220
eval_bleu: 13.5479
eval_meteor: 0.3804
eval_rougeL: 0.3445
eval_cider: 0.2929
eval_runtime: 420.6666
eval_samples_per_second: 2.3770
eval_steps_per_second: 0.2970


In [25]:
# first_block = model.base_model.model.transformer.h[0]         # First transformer block
# attn = first_block.attn                                       # Attention module
# c_attn = attn.c_attn                                          # The fused QKV projection
# print(f"c_attn.weight shape: {c_attn.weight.shape}")          # Should be [3072, 1024]